# Security Policies LLM

In [1]:
from IPython.display import display, HTML

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "utils"))

%load_ext autoreload
%autoreload 2

from read import read_pdf
from llm import get_policies, get_embedding, get_dependencies, get_workflows
from temp import write_to_temp, read_from_temp
from db import add_article, search, find_clauses
from text import create_workflow_input

In [3]:
load_data = False

### Read PDF file contents

In [4]:
pdf = read_pdf("../data/security_policies.pdf")
pdf_contents = "\n".join(pdf)

display(HTML(pdf_contents[:495]))

### Extract policies

In [5]:
filepath = "out.pickle"

In [6]:
if load_data:
    policies = get_policies(content=pdf_contents)
    write_to_temp(filepath, policies)

In [7]:
policies = read_from_temp(filepath)

for article in policies.articles:
    display(HTML(f"<strong>Article {article.number}:</strong> {article.title}"))

## Constructing Retrieval-augmented generation

In [8]:
get_embedding(policies.articles[0].clauses[0].text)[:10]

[-0.027899671345949173,
 0.007937368005514145,
 -0.047299642115831375,
 0.015589096583425999,
 -0.0610421746969223,
 0.011482739821076393,
 0.012260657735168934,
 0.040641698986291885,
 0.0211750790476799,
 0.008149749599397182]

In [9]:
if load_data:
    for article in policies.articles:
        add_article(article)

In [10]:
query = "Automatic deletion of user data"

results = search(query)

for result in results:
    display(HTML(f"<strong>Article {result[0]}</strong><br /> {result[1]}. {result[2]}"))
    display(HTML("<br />"))

### Get dependent articles

In [11]:
selected = results[0]
dependencies = get_dependencies(selected[2])

In [12]:
display(HTML(f"Article {selected[0]}({selected[1]}) is dependent on:"))

for dependency in dependencies.dependencies:
    display(HTML(f"<li>Article {dependency.article} ({dependency.clause})"))

In [13]:
all_clauses = [selected, *find_clauses(dependencies)]

workflow_input = create_workflow_input(all_clauses)

In [20]:
all_clauses

[(17,
  1,
  'The data subject shall have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay and the controller shall have the obligation to erase personal data without undue delay where one of the following grounds applies:(a) the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed;(b) the data subject withdraws consent on which the processing is based according to point (a) of Article 6(1), or point (a) of Article 9(2), and where there is no other legal ground for the processing;(c) the data subject objects to the processing pursuant to Article 21(1) and there are no overriding legitimate grounds for the processing, or the data subject objects to the processing pursuant to Article 21(2);(d) the personal data have been unlawfully processed;(e) the personal data have to be erased for compliance with a legal obligation in Union or Member State law to which the c

In [19]:
workflow = get_workflows(workflow_input)

for workflow in workflow.steps:
    display(HTML(f"<li>{workflow}"))